[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/solutions/63_mcts_search_solution.ipynb)

# 🔴 Solution: MCTS Search Step

Reference solution for a single Monte Carlo Tree Search step using the PUCT (Predictor + UCT) selection formula, as used in AlphaGo/AlphaZero and modern LLM reasoning systems.

$$\text{UCB}(i) = V_i + c_{\text{puct}} \cdot \sqrt{\frac{\ln(N_{\text{parent}} + 1)}{N_i + 1}}$$

In [ ]:
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass

In [ ]:
import torch

In [ ]:
# ✅ SOLUTION

def mcts_step(values, visit_counts, parent_visits, c_puct=1.414):
    import math as _math
    exploration = c_puct * torch.sqrt(
        torch.log(torch.tensor(parent_visits + 1, dtype=torch.float32)) /
        (visit_counts.float() + 1)
    )
    ucb = values + exploration
    selected_idx = int(ucb.argmax().item())
    rollout_value = torch.rand(1).item()
    n = visit_counts[selected_idx].item()
    new_value = (values[selected_idx].item() * n + rollout_value) / (n + 1)
    updated_values = values.clone()
    updated_visits = visit_counts.clone()
    updated_values[selected_idx] = new_value
    updated_visits[selected_idx] += 1
    return selected_idx, updated_values, updated_visits

In [ ]:
torch.manual_seed(42)
num_children = 5
values = torch.zeros(num_children)
visits = torch.zeros(num_children, dtype=torch.long)
parent_visits = 0

print("Running 5 MCTS steps:")
print(f"{'Step':>4}  {'Selected':>8}  {'Visit counts'}")
for step in range(5):
    idx, values, visits = mcts_step(values, visits, parent_visits)
    parent_visits += 1
    print(f"{step+1:>4}  {idx:>8}  {visits.tolist()}")

print(f"\nTotal visits: {visits.sum().item()} (expected 5)")
print(f"All nodes visited at least once: {(visits > 0).all().item()}")

In [ ]:
from torch_judge import check
check("mcts_search")